In [ ]:
# ============================================================
# Cell 1: Setup
# ============================================================
import os, sys, torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision.utils import save_image

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Add project utils to path
PROJECT_ROOT = '/content/drive/MyDrive/Project2'
sys.path.insert(0, PROJECT_ROOT)

# Clone CycleGAN repo if needed
if not os.path.exists('pytorch-CycleGAN-and-pix2pix'):
    !git clone https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix.git
    !pip install -q visdom dominate

SAVE_DIR = os.path.join(PROJECT_ROOT, 'results/svhn_mnist')
os.makedirs(SAVE_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'Save dir: {SAVE_DIR}')

In [ ]:
# === Quick Test Mode ===
# Set FAST_TEST = True to verify the notebook runs end-to-end (~5 min)
# Set FAST_TEST = False for full experiment
FAST_TEST = True

if FAST_TEST:
    CYCLEGAN_EPOCHS = 1
    CYCLEGAN_DECAY = 0
    CYCADA_EPOCHS = 1
    CLASSIFIER_EPOCHS = 2
    MAX_IMAGES = 100
    NUM_TEST = 50
else:
    CYCLEGAN_EPOCHS = 50
    CYCLEGAN_DECAY = 50
    CYCADA_EPOCHS = 50
    CLASSIFIER_EPOCHS = 20
    MAX_IMAGES = 5000
    NUM_TEST = 60000

print(f'FAST_TEST={FAST_TEST}: epochs={CYCLEGAN_EPOCHS}+{CYCLEGAN_DECAY}, images={MAX_IMAGES}')

In [ ]:
# ============================================================
# Cell 2: Load Data
# ============================================================
from utils.data_utils import get_svhn, get_mnist, get_paired_loaders

svhn_train = get_svhn(train=True, img_size=32)   # returns grayscale
svhn_test = get_svhn(train=False, img_size=32)
mnist_train = get_mnist(train=True, img_size=32)
mnist_test = get_mnist(train=False, img_size=32)

source_loader, target_loader = get_paired_loaders(svhn_train, mnist_train, batch_size=64)
target_test_loader = torch.utils.data.DataLoader(mnist_test, batch_size=64, shuffle=False)
print(f"SVHN train: {len(svhn_train)}, MNIST train: {len(mnist_train)}, MNIST test: {len(mnist_test)}")

# Task I: Style Transfer Comparison

In [ ]:
# ============================================================
# Cell 4: Save images for CycleGAN training
# ============================================================
def save_dataset_images(dataset, save_dir, max_images=None):
    """Save dataset images as individual PNG files for CycleGAN."""
    os.makedirs(save_dir, exist_ok=True)
    n = len(dataset) if max_images is None else min(max_images, len(dataset))
    for i in range(n):
        img, _ = dataset[i]
        save_image(img, os.path.join(save_dir, f'{i:06d}.png'))
    print(f'Saved {n} images to {save_dir}')

# Save SVHN (source) and MNIST (target) images
save_dataset_images(svhn_train, '/content/cyclegan_data/svhn2mnist/trainA', max_images=MAX_IMAGES)
save_dataset_images(mnist_train, '/content/cyclegan_data/svhn2mnist/trainB', max_images=MAX_IMAGES)


In [ ]:
# ============================================================
# Cell 5: Train Standard CycleGAN (spatial)
# ============================================================
from utils.cyclegan_wrapper import get_cyclegan_train_cmd

cmd = get_cyclegan_train_cmd(
    n_epochs=CYCLEGAN_EPOCHS,
    n_epochs_decay=CYCLEGAN_DECAY,
    dataroot='/content/cyclegan_data/svhn2mnist',
    name='svhn2mnist_spatial',
    load_size=32,
    crop_size=32,
    input_nc=1,
    output_nc=1,
)
print(f'Running: {cmd}')
!{cmd}

In [ ]:
# ============================================================
# Cell 6: Train Spectral CycleGAN
# ============================================================
from utils.spectral_cyclegan import save_low_freq_images, reconstruct_from_translated_low

# Decompose source images into low/high frequency
src_low_dir, src_high_dir = save_low_freq_images(
    '/content/cyclegan_data/svhn2mnist/trainA',
    '/content/cyclegan_data/svhn2mnist_spectral/source_decomposed',
    beta=0.03, mode='L'
)
tgt_low_dir, tgt_high_dir = save_low_freq_images(
    '/content/cyclegan_data/svhn2mnist/trainB',
    '/content/cyclegan_data/svhn2mnist_spectral/target_decomposed',
    beta=0.03, mode='L'
)

# Prepare low-freq images for CycleGAN
from utils.cyclegan_wrapper import prepare_cyclegan_data
prepare_cyclegan_data(src_low_dir, tgt_low_dir, '/content/cyclegan_data/svhn2mnist_spectral_lowfreq')

# Train CycleGAN on low-freq only
cmd_spectral = get_cyclegan_train_cmd(
    n_epochs=CYCLEGAN_EPOCHS,
    n_epochs_decay=CYCLEGAN_DECAY,
    dataroot='/content/cyclegan_data/svhn2mnist_spectral_lowfreq',
    name='svhn2mnist_spectral',
    load_size=32,
    crop_size=32,
    input_nc=1,
    output_nc=1,
)
print(f'Running: {cmd_spectral}')
!{cmd_spectral}

In [ ]:
import glob, shutil, os

for src, dst in [
    ('svhn2mnist/trainA', 'svhn2mnist/testA'),
    ('svhn2mnist/trainB', 'svhn2mnist/testB'),
    ('svhn2mnist_spectral_lowfreq/trainA', 'svhn2mnist_spectral_lowfreq/testA'),
    ('svhn2mnist_spectral_lowfreq/trainB', 'svhn2mnist_spectral_lowfreq/testB'),
]:
    src_dir = f'/content/cyclegan_data/{src}'
    dst_dir = f'/content/cyclegan_data/{dst}'
    os.makedirs(dst_dir, exist_ok=True)
    for f in sorted(glob.glob(f'{src_dir}/*.png'))[:50]:
        shutil.copy2(f, dst_dir)
    print(f"Copied {len(os.listdir(dst_dir))} images to {dst_dir}")

In [ ]:
# ============================================================
# Cell 7: Task I Visualization
# ============================================================
from utils.viz_utils import show_image_grid
from utils.cyclegan_wrapper import get_cyclegan_test_cmd

# Run CycleGAN inference (spatial)
cmd_test_spatial = get_cyclegan_test_cmd(
    dataroot='/content/cyclegan_data/svhn2mnist',
    name='svhn2mnist_spatial',
    load_size=32,
    crop_size=32,
    input_nc=1,
    output_nc=1,
)
!{cmd_test_spatial}

# Run CycleGAN inference (spectral low-freq)
cmd_test_spectral = get_cyclegan_test_cmd(
    dataroot='/content/cyclegan_data/svhn2mnist_spectral_lowfreq',
    name='svhn2mnist_spectral',
    load_size=32,
    crop_size=32,
    input_nc=1,
    output_nc=1,
)
!{cmd_test_spectral}

# Load and visualize results
from torchvision import transforms
from PIL import Image
import glob

to_tensor = transforms.ToTensor()
n_show = 8

# Original SVHN images
original_imgs = torch.stack([svhn_train[i][0] for i in range(n_show)])

# Spatial CycleGAN translated
spatial_dir = 'results/svhn2mnist_spatial/test_latest/images'
spatial_files = sorted(glob.glob(os.path.join(spatial_dir, '*_fake_B.png')))[:n_show]
spatial_imgs = torch.stack([to_tensor(Image.open(f).convert('L')) for f in spatial_files])

# Spectral CycleGAN: reconstruct translated low-freq + original high-freq
spectral_dir = 'results/svhn2mnist_spectral/test_latest/images'
spectral_files = sorted(glob.glob(os.path.join(spectral_dir, '*_fake_B.png')))[:n_show]
if len(spectral_files) > 0:
    spectral_low_imgs = torch.stack([to_tensor(Image.open(f).convert('L')) for f in spectral_files])
    high_freq_files = sorted(os.listdir(src_high_dir))[:n_show]
    high_imgs = torch.stack([to_tensor(Image.open(os.path.join(src_high_dir, f)).convert('L')) for f in high_freq_files])
    from utils.fft_utils import spectral_reconstruct
    spectral_imgs = spectral_reconstruct(spectral_low_imgs, high_imgs)
else:
    spectral_imgs = original_imgs  # fallback

show_image_grid(
    {
        'Original SVHN': original_imgs,
        'Spatial CycleGAN': spatial_imgs,
        'Spectral CycleGAN': spectral_imgs,
    },
    nrow=n_show,
    title='Task I: SVHN → MNIST Style Transfer Comparison',
    save_path=os.path.join(SAVE_DIR, 'task1_style_transfer.png'),
)

# Task II: UDA Benchmark

In [ ]:
# ============================================================
# Cell 9: Source-Only Baseline
# ============================================================
from utils.classifiers import LeNet5, train_classifier, evaluate_classifier

results = {}

# Train LeNet5 on SVHN (source only)
model_source = LeNet5(num_classes=10, in_channels=1).to(device)
model_source = train_classifier(model_source, source_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device)

# Evaluate on MNIST (target)
acc_source_only = evaluate_classifier(model_source, target_test_loader, device=device)
results['Source Only'] = acc_source_only

In [ ]:
# ============================================================
# Cell 10: CycleGAN UDA
# ============================================================

# Translate SVHN images with spatial CycleGAN
from PIL import Image
from torchvision import transforms

# Load the trained CycleGAN generator
# Use the test results from spatial CycleGAN
translated_dir = 'results/svhn2mnist_spatial/test_latest/images'
to_tensor = transforms.ToTensor()

# Build a dataset from translated images
class TranslatedDataset(torch.utils.data.Dataset):
    def __init__(self, image_dir, original_dataset, suffix='_fake_B.png'):
        self.image_dir = image_dir
        self.files = sorted(glob.glob(os.path.join(image_dir, f'*{suffix}')))
        self.original_dataset = original_dataset
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize(32),
            transforms.ToTensor(),
        ])
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        img = Image.open(self.files[idx])
        img = self.transform(img)
        _, label = self.original_dataset[idx]
        return img, label

import glob
cyclegan_dataset = TranslatedDataset(translated_dir, svhn_train)
cyclegan_loader = torch.utils.data.DataLoader(cyclegan_dataset, batch_size=64, shuffle=True)

model_cyclegan = LeNet5(num_classes=10, in_channels=1).to(device)
model_cyclegan = train_classifier(model_cyclegan, cyclegan_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device)
acc_cyclegan = evaluate_classifier(model_cyclegan, target_test_loader, device=device)
results['CycleGAN'] = acc_cyclegan

In [ ]:
# ============================================================
# Cell 11: Spectral CycleGAN UDA
# ============================================================

# Reconstruct full images from translated low-freq + original high-freq
spectral_translated_dir = 'results/svhn2mnist_spectral/test_latest/images'
spectral_recon_dir = '/content/cyclegan_data/svhn2mnist_spectral_reconstructed'

# Get translated low-freq directory
translated_low_files = sorted(glob.glob(os.path.join(spectral_translated_dir, '*_fake_B.png')))
translated_low_dir_tmp = '/content/cyclegan_data/svhn2mnist_spectral_translated_low'
os.makedirs(translated_low_dir_tmp, exist_ok=True)
for i, f in enumerate(translated_low_files):
    import shutil
    shutil.copy(f, os.path.join(translated_low_dir_tmp, f'{i:06d}.png'))

# Reconstruct
reconstruct_from_translated_low(translated_low_dir_tmp, src_high_dir, spectral_recon_dir)

# Build dataset from reconstructed images
class ReconstructedDataset(torch.utils.data.Dataset):
    def __init__(self, image_dir, original_dataset):
        self.image_dir = image_dir
        self.files = sorted([f for f in os.listdir(image_dir) if f.endswith('.png')])
        self.original_dataset = original_dataset
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize(32),
            transforms.ToTensor(),
        ])
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.image_dir, self.files[idx]))
        img = self.transform(img)
        _, label = self.original_dataset[idx]
        return img, label

spectral_dataset = ReconstructedDataset(spectral_recon_dir, svhn_train)
spectral_loader = torch.utils.data.DataLoader(spectral_dataset, batch_size=64, shuffle=True)

model_spectral = LeNet5(num_classes=10, in_channels=1).to(device)
model_spectral = train_classifier(model_spectral, spectral_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device)
acc_spectral = evaluate_classifier(model_spectral, target_test_loader, device=device)
results['Spectral CycleGAN'] = acc_spectral

In [ ]:
# ============================================================
# Cell 12: CyCADA
# ============================================================
from utils.cycada import train_cycada, translate_with_cycada

# Train CyCADA using source-only model as task classifier
G_s2t, G_t2s = train_cycada(
    source_loader, target_loader, model_source,
    num_classes=10, in_channels=1,
    n_epochs=CYCADA_EPOCHS, lr=2e-4,
    lambda_cyc=10.0, lambda_task=1.0,
    device=device,
)

# Translate source images to target domain
translated_data = translate_with_cycada(G_s2t, source_loader, device=device)

# Create DataLoader from translated data
class CycadaDataset(torch.utils.data.Dataset):
    def __init__(self, translated_data):
        self.images = torch.cat([imgs for imgs, _ in translated_data], dim=0)
        self.labels = torch.cat([labels for _, labels in translated_data], dim=0)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

cycada_dataset = CycadaDataset(translated_data)
cycada_loader = torch.utils.data.DataLoader(cycada_dataset, batch_size=64, shuffle=True)

model_cycada = LeNet5(num_classes=10, in_channels=1).to(device)
model_cycada = train_classifier(model_cycada, cycada_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device)
acc_cycada = evaluate_classifier(model_cycada, target_test_loader, device=device)
results['CyCADA'] = acc_cycada

In [ ]:
# ============================================================
# Cell 13: FDA (Fourier Domain Adaptation)
# ============================================================
from utils.fft_utils import fda_transfer

# Apply FDA: transfer low-freq amplitude from MNIST (target) to SVHN (source)
class FDADataset(torch.utils.data.Dataset):
    def __init__(self, source_loader, target_loader, beta=0.01):
        self.source_images = []
        self.source_labels = []
        target_iter = iter(target_loader)
        for src_imgs, src_labels in source_loader:
            try:
                tgt_imgs, _ = next(target_iter)
            except StopIteration:
                target_iter = iter(target_loader)
                tgt_imgs, _ = next(target_iter)
            # Match batch sizes
            bs = min(src_imgs.size(0), tgt_imgs.size(0))
            transferred = fda_transfer(src_imgs[:bs], tgt_imgs[:bs], beta=beta)
            self.source_images.append(transferred)
            self.source_labels.append(src_labels[:bs])
        self.source_images = torch.cat(self.source_images, dim=0)
        self.source_labels = torch.cat(self.source_labels, dim=0)

    def __len__(self):
        return len(self.source_labels)

    def __getitem__(self, idx):
        return self.source_images[idx], self.source_labels[idx]

fda_dataset = FDADataset(source_loader, target_loader, beta=0.01)
fda_loader = torch.utils.data.DataLoader(fda_dataset, batch_size=64, shuffle=True)

model_fda = LeNet5(num_classes=10, in_channels=1).to(device)
model_fda = train_classifier(model_fda, fda_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device)
acc_fda = evaluate_classifier(model_fda, target_test_loader, device=device)
results['FDA'] = acc_fda

In [ ]:
# ============================================================
# Cell 14: Results Summary
# ============================================================
from utils.viz_utils import print_results_table
from utils.eval_utils import save_results

print_results_table(results, 'SVHN → MNIST')
save_results(results, 'SVHN_to_MNIST', save_dir=SAVE_DIR)